# A straight line through living area

`GrLivArea` is the above-grade living area. On all 1460 rows it is the sum of the two floors and the low-quality finished area. 829 houses have a second floor of 0.

Fit a line on a toy first, in units a person can check. Let $x$ be living area in hundreds of square feet, and let $y$ be price in thousands of dollars. The three houses are $(10, 100)$, $(20, 200)$, and $(30, 240)$.

$$
\begin{aligned}
\bar x &= 20, \qquad \bar y = 180 \\
\sum (x_i - \bar x)^2 &= (-10)^2 + 0^2 + 10^2 = 200 \\
\sum (x_i - \bar x)(y_i - \bar y) &= (-10)(-80) + 0 \cdot 20 + 10 \cdot 60 = 1400 \\
b_1 &= \dfrac{1400}{200} = 7 \\
b_0 &= 180 - 7 \cdot 20 = 40
\end{aligned}
$$

The line is $y = 40 + 7x$. The fitted prices are 110, 180, and 250. The residuals are $-10$, $20$, and $-10$.

$$
(-10)^2 + 20^2 + (-10)^2 = 600.
$$

The same two sums, on the Ames file, use $x$ in square feet and $y$ in dollars:

$$
n = 1460, \qquad \sum x = 2212577, \qquad \sum y = 264144946.
$$

The slope and intercept are then one fraction each. The slope is about 107 dollars per square foot, and the intercept is about 18569 dollars. For house 1, $x = 1710$, the line returns about 201762 dollars. The house sold for 208500.


In [2]:
# Locate the contest tables beside this notebook, or under the course root.
from pathlib import Path
import csv

def manifest_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "15-House Prices" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "data_description.txt").exists():
            return direct
    raise FileNotFoundError(name)

def read_manifest(name):
    with manifest_path(name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

train = read_manifest("train.csv")
test = read_manifest("test.csv")
submission = read_manifest("sample_submission.csv")
print(len(train), len(test), len(submission))
print(train[0]["Id"], train[0]["SalePrice"])


1460 1459 1459
1 208500


In [3]:
# The toy line, then the same formulas on all 1460 houses.
from fractions import Fraction
toy_x = [10, 20, 30]
toy_y = [100, 200, 240]
x_bar = Fraction(sum(toy_x), 3)
y_bar = Fraction(sum(toy_y), 3)
sxx = sum((x - x_bar) ** 2 for x in toy_x)
sxy = sum((x - x_bar) * (y - y_bar) for x, y in zip(toy_x, toy_y))
slope = sxy / sxx
intercept = y_bar - slope * x_bar
fitted = [intercept + slope * x for x in toy_x]
residuals = [y - pred for y, pred in zip(toy_y, fitted)]
rss = sum(residual ** 2 for residual in residuals)
print(slope, intercept, fitted, rss)
assert x_bar == 20 and y_bar == 180
assert sxx == 200 and sxy == 1400
assert slope == 7 and intercept == 40
assert fitted == [110, 180, 250]
assert rss == 600

areas = [int(row["GrLivArea"]) for row in train]
prices = [int(row["SalePrice"]) for row in train]
count = len(train)
sum_x = sum(areas)
sum_y = sum(prices)
sum_xx = sum(area * area for area in areas)
sum_xy = sum(area * price for area, price in zip(areas, prices))
ames_slope = Fraction(
    count * sum_xy - sum_x * sum_y,
    count * sum_xx - sum_x * sum_x,
)
ames_intercept = Fraction(sum_y, count) - ames_slope * Fraction(sum_x, count)
house_fit = ames_intercept + ames_slope * 1710
identity = sum(
    int(row["1stFlrSF"]) + int(row["2ndFlrSF"]) + int(row["LowQualFinSF"])
    == int(row["GrLivArea"])
    for row in train
)
print(sum_x, sum_y)
print(ames_slope)
print(ames_intercept)
print(float(house_fit))
assert identity == 1460
assert sum(int(row["2ndFlrSF"]) == 0 for row in train) == 829
assert sum_x == 2212577
assert sum_y == 264144946
assert ames_slope == Fraction(63013517624438, 588194777211)
assert ames_intercept == Fraction(10922204026681828, 588194777211)
assert abs(float(house_fit) - 201761.93968804833) < 1e-6


7 40 [Fraction(110, 1), Fraction(180, 1), Fraction(250, 1)] 600
2212577 264144946
63013517624438/588194777211
10922204026681828/588194777211
201761.93968804833


## Pitfall

A slope of about 107 dollars per square foot describes the cloud. It does not say that every extra square foot was sold for 107 dollars. House 1's own gap is $208500 - 201762$, about 6700 dollars, and later lessons meet houses whose gaps are hundreds of thousands.

## Summary

On the toy, the line is $40 + 7x$ and the residual sum of squares is 600. On the file, the living areas sum to 2212577, the prices sum to 264144946, and house 1 is fitted at about 201762 against a sale of 208500.
